In [ ]:
# import libraries
import geopandas as gpd
from pystac import Collection
import pystac
import xarray as xr
import shapely
import json
from datetime import datetime
from glob import glob
from shapely.geometry import box, mapping

Create the main collection to which the files/STAC items will be added.

In [ ]:
collectionid = "lightning-clusters"
description = "GOES-GLM to EarthCARE collocated data"
collection = Collection.from_dict(
    
{
  "type": "Collection",
  "id": collectionid,
  "stac_version": "1.1.0",
  "title": "EarthCARE Lightning Clusters",
  "description": description,
  "extent": {
    "spatial": {
      "bbox": [
        [-180, 
         -90, 
         180,
         90]
      ]
    },
    "temporal": {
      "interval": [
        [
          "2024-08-01T00:00:00Z",
          "2025-12-31T00:00:00Z"
        ]
      ]
    }
  },
  "license": "CC-BY-4.0",
  "links": []

}

)

collection.extra_fields.update({
  'institution': 'European Space Agency (ESA)',
  'history': 'Created on 2026-01-28',
  'source': 'EarthCARE MSI/CPR + NOAA GOES Geostationary Lightning Mapper (GLM) L2 LCFA (platform: GOES-16)',
  'references': 'https://www.goes-r.gov/spacesegment/glm.html, https://earth.esa.int/eogateway/missions/earthcare'}
)
collection

<Collection id=lightning-clusters>

In [ ]:
collection.validate()

['https://schemas.stacspec.org/v1.1.0/collection-spec/json-schema/collection.json']

Extract all variable metadata from sample nc files.

In [22]:
glm = xr.open_dataset("/Volumes/T7/data/blanka/C1/GLM_20240801T041007Z_20240801T051832Z_33_0_01000B.nc")
glm_vars = {}
for var in glm.variables:
    glm_vars[var] = glm[var].attrs
li = xr.open_dataset("/Volumes/T7/data/blanka/C1/LI_20240801T005006Z_20240801T031005Z_67235_193_00998A.nc")
li_vars = {}
for var in li.variables:
    li_vars[var] = li[var].attrs
c2li = xr.open_dataset("/Volumes/T7/data/blanka/C2/CPR-LI-sum_20240801T015231Z_20240801T020411Z_193_00998A.nc")
c2li_vars = {}
for var in c2li.variables:
    c2li_vars[var] = c2li[var].attrs
c2glm = xr.open_dataset("/Volumes/T7/data/blanka/C2/CPR-GLM-sum_20250709T060737Z_20250709T061928Z_215_06322B.nc")
c2glm_vars = {}
for var in c2glm.variables:
    c2glm_vars[var] = c2glm[var].attrs


In [23]:
c3 = {
    "title": "EarthCARE lightning storm catalogue",
    "description": "Each catalogue entry represents an individual lightning cluster associated with an EarthCARE CPR overpass, identified within \u00b12.5 minutes of the overpass and within a 2.5 km radius of the CPR nadir track.",
    "unique_id": "Unique identifier of the lightning cluster (orbit_frame + source + cluster_id)",  
    "orbit_frame": "Orbit/frame identifier of the EarthCARE overpass",
    "source": "Lightning data source: LI (MTG) or GLM (GOES)",
    "parent_cluster_id": "Parent lightning cluster identifier",
    "cluster_id": "Lightning cluster identifier",
    "surface_type": "Surface classification at cluster location (land, water, coast)",
    "peak_datetime": "Time of CPR sample with maximum lightning activity",
    "peak_lat": "Latitude of CPR sample with maximum lightning activity",
    "peak_lon": "Longitude of CPR sample with maximum lightning activity",
    "peak_lightning": "Maximum lightning group count at a CPR sample",
    "nadir_lightning": "Lightning group count within \u00b12.5 min and \u00b12.5 km of CPR nadir track",
    "cluster_lightning": "Lightning group count within \u00b12.5 min of CPR peak time (any distance)",
    "cluster_area_km2": "Area of cluster in km\u00b2",
    "cluster_mean_lat": "Mean latitude of lightning groups within \u00b12.5 min of CPR peak time",
    "cluster_mean_lon": "Mean longitude of lightning groups within \u00b12.5 min of CPR peak time",
    "cluster_dist_km": "Minimum distance (km) between CPR track and mean cluster location",
    "first_lightning_min": "Minute offset of first lightning in parent cluster relative to peak_datetime",
    "last_lightning_min": "Minute offset of last lightning in parent cluster relative to peak_datetime",
    "duration_min": "Parent cluster duration in minutes",
    "travel_km": "Shortest distance (km) between first and last lightning locations in parent cluster",
    "minute_counts": "Lightning group counts in parent cluster per minute (\u201360\u2026+60) relative to peak_datetime",
    "minute_mean_lat": "Mean lightning latitude in parent cluster per minute (\u201360\u2026+60) relative to peak_datetime",
    "minute_mean_lon": "Mean lightning longitude in parent cluster per minute (\u201360\u2026+60) relative to peak_datetime",
    "missing_peak_minutes": "Total minutes of missing data overlapping \u00b12.5 min around peak_datetime",
    "masked_minute_counts": "Lightning group counts per minute (\u201360\u2026+60) with missing intervals masked as null"
  }

In [25]:
c3evol = gpd.read_parquet("/Volumes/T7/data/blanka/C3_parquet/gdf_evolution.parquet")
c3summary = gpd.read_parquet("/Volumes/T7/data/blanka/C3_parquet/gdf_summary.parquet")
list(c3evol.columns)
list(c3summary.columns)
c3evol_descriptions = {}
for col in c3evol.columns:
    if col in c3:
        c3evol_descriptions[col] = c3[col]
    
c3summary_descriptions = {}
for col in c3summary.columns:
    if col in c3:
        c3summary_descriptions[col] = c3[col]
    

## Generate stac items per file type and add to the collection.

In [ ]:
c1_li_files = glob("/Volumes/T7/data/blanka/C1_parquet/LI*.parquet")
c1_glm_files = glob("/Volumes/T7/data/blanka/C1_parquet/GLM*.parquet")

c2_glm_files = glob("/Volumes/T7/data/blanka/C2_parquet/GLM_C2*.parquet")
c2_li_files = glob("/Volumes/T7/data/blanka/C2_parquet/LI_C2*.parquet")

c3evol_files = glob("/Volumes/T7/data/blanka/C3_parquet/gdf_evolution.parquet")
c3summary_files = glob("/Volumes/T7/data/blanka/C3_parquet/gdf_summary.parquet")


{'lightning_count_2p5': {'long_name': 'Lightning groups count per cluster within 2.5 km radius and ±2.5 min time window around each CPR sample',
  'units': '1'},
 'lightning_count_5': {'long_name': 'Lightning groups count per cluster within 5 km radius and ±5 min time window around each CPR sample',
  'units': '1'},
 'cpr': {'long_name': 'EarthCARE CPR sample index'},
 'cluster_id': {'long_name': 'Cluster identifier'},
 'latitude': {'long_name': 'CPR nadir latitude', 'units': 'degrees_north'},
 'longitude': {'long_name': 'CPR nadir longitude', 'units': 'degrees_east'},
 'time': {'long_name': 'CPR observation time'},
 'land_flag': {'long_name': 'CPR land/water flag',
  'definition': '1 = land, 0 = not land',
  'units': '1'}}

In [28]:
prefix = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/storm-data/"

import os
from datetime import timezone

In [ ]:
## crete STAC items for teh C1 LI files collection

for file in c1_li_files:

    base_name = os.path.basename(file)
    item_id = os.path.splitext(base_name)[0] 
    gdf = gpd.read_parquet(file)
    bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]
    geom = mapping(box(*bounds))

    item_datetime = datetime.now(timezone.utc)

    properties = {
        "parquet:columns": li_vars,
        "source": "LI",
        "type": "Lightning Strikes"
    }

    # Create the STAC Item
    item = pystac.Item(
        id=item_id,
        geometry=geom,
        bbox=bounds,
        datetime=item_datetime,
        properties=properties
    )

    # add the asset
    path = prefix + file.split("/")[-1]
    asset_href = path
    asset = pystac.Asset(
        href=asset_href,
        media_type="application/x-parquet",
        roles=["data"]
    )

    item.add_asset(key="data", asset=asset) 
    collection.add_item(item)
    print(f"  - Added Item: {item_id}")


  - Added Item: LI_2024_8
  - Added Item: LI_2024_9
  - Added Item: LI_2024_10
  - Added Item: LI_2024_11
  - Added Item: LI_2024_12
  - Added Item: LI_2025_1
  - Added Item: LI_2025_2
  - Added Item: LI_2025_3
  - Added Item: LI_2025_4
  - Added Item: LI_2025_5
  - Added Item: LI_2025_6
  - Added Item: LI_2025_7
  - Added Item: LI_2025_8
  - Added Item: LI_2025_9
  - Added Item: LI_2025_10
  - Added Item: LI_2025_11
  - Added Item: LI_2025_12
  - Added Item: LI_2026_1


In [ ]:
## crete STAC items for teh C1 GLM files collection

for file in c1_glm_files:

    base_name = os.path.basename(file)
    item_id = os.path.splitext(base_name)[0] 
    gdf = gpd.read_parquet(file)
    bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]
    geom = mapping(box(*bounds))

    item_datetime = datetime.now(timezone.utc)

    properties = {
        "parquet:columns": glm_vars,
        "source": "GLM",
        "type": "Lightning Strikes"

    }

    item = pystac.Item(
        id=item_id,
        geometry=geom,
        bbox=bounds,
        datetime=item_datetime,
        properties=properties
    )

    path = prefix + file.split("/")[-1]
    asset_href = path
    asset = pystac.Asset(
        href=asset_href,
        media_type="application/x-parquet",
        roles=["data"]
    )
    
    item.add_asset(key="data", asset=asset) 
    collection.add_item(item)
    print(f"  - Added Item: {item_id}")


  - Added Item: GLM_2024_8
  - Added Item: GLM_2024_9
  - Added Item: GLM_2024_10
  - Added Item: GLM_2024_11
  - Added Item: GLM_2024_12
  - Added Item: GLM_2025_1
  - Added Item: GLM_2025_2
  - Added Item: GLM_2025_3
  - Added Item: GLM_2025_4
  - Added Item: GLM_2025_5
  - Added Item: GLM_2025_6
  - Added Item: GLM_2025_7
  - Added Item: GLM_2025_8
  - Added Item: GLM_2025_9
  - Added Item: GLM_2025_10
  - Added Item: GLM_2025_11
  - Added Item: GLM_2025_12
  - Added Item: GLM_2026_1


Add the collection 2 files individually

In [ ]:
file = c2_glm_files[0]
base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 

gdf = gpd.read_parquet(file)
bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]
geom = mapping(box(*bounds))
item_datetime = datetime.now(timezone.utc)

properties = {
    "parquet:columns": c2glm_vars,
    "source": "GLM",
    "type": "Lightning Strikes along EarthCARE nadir track"

}

item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=item_datetime,
    properties=properties
)
path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset)
collection.add_item(item)
print(f"  - Added Item: {item_id}")

  - Added Item: GLM_C2


In [ ]:
file = c2_li_files[0]

base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 
gdf = gpd.read_parquet(file)
bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]

geom = mapping(box(*bounds))
item_datetime = datetime.now(timezone.utc)
properties = {
    "parquet:columns": c2li_vars,
    "source": "LI",
    "type": "Lightning Strikes along EarthCARE nadir track"

}

item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=item_datetime,
    properties=properties
)

path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset) 
collection.add_item(item)
print(f"  - Added Item: {item_id}")

  - Added Item: LI_C2


Add the summary and the evolution files

In [ ]:
file = c3summary_files[0]

base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 
gdf = gpd.read_parquet(file)
bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]
geom = mapping(box(*bounds))
item_datetime = datetime.now(timezone.utc)
properties = {
    "parquet:columns": c3summary_descriptions,
    "type": "Storm catalogue summary statistics"

}
item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=item_datetime,
    properties=properties
)

path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset) 
collection.add_item(item)
print(f"  - Added Item: {item_id}")

  - Added Item: gdf_summary


In [ ]:
file = c3evol_files[0]

base_name = os.path.basename(file)
item_id = os.path.splitext(base_name)[0] 
gdf = gpd.read_parquet(file)
bounds = list(gdf.total_bounds) # [minx, miny, maxx, maxy]
geom = mapping(box(*bounds))
item_datetime = datetime.now(timezone.utc)
properties = {
    "parquet:columns": c3evol_descriptions,
    "type": "Storm catalogue evolution statistics"

}
item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bounds,
    datetime=item_datetime,
    properties=properties
)

path = prefix + file.split("/")[-1]
asset_href = path
asset = pystac.Asset(
    href=asset_href,
    media_type="application/x-parquet",
    roles=["data"]
)

item.add_asset(key="data", asset=asset) 
collection.add_item(item)
print(f"  - Added Item: {item_id}")

  - Added Item: gdf_evolution


In [36]:
collection

<Collection id=lightning-clusters>

In [ ]:
# save the catalog
collection.normalize_and_save(
    root_href='./storm-data/'
    catalog_type=pystac.CatalogType.SELF_CONTAINED
)

